In [ ]:
#imports
import torch 
from torch.utils.data import Dataset, DataLoader
from torch import nn
import os
import numpy as np
import json
from pathlib import Path
import gc
import hashlib

EPOCHS = 100
DEPTH = 10
OPTIMIZER = torch.optim.AdamW
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-3
DROPOUT = .2
LARGEST_PATCH = 100
TWO_WINDOW_MAX_SIZE = 11
THREE_WINDOW_MAX_SIZE = 7
NUM_GPUS = torch.cuda.device_count()

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
def pad_to_size(x, target=100):
  return nn.functional.pad(x, (0, target - x.shape[-1], 0, target - x.shape[-2]))

In [ ]:
def save_results(
    results_dir: str,
    tag: str,
    history: list,
    preds,
    targets
):
  os.makedirs(results_dir, exist_ok=True)

  torch.save({
        'predictions': torch.cat(preds, dim=0).cpu(), 
        'targets': torch.cat(targets, dim=0).cpu()
      },os.path.join(results_dir, f'predictions_{tag}.pt'))

  with open(os.path.join(results_dir, f"results_{tag}.json"), "w") as fp:
    json.dump(history, fp)

In [30]:
#Data Preparation
class PhantomDataset(Dataset):
  def __init__(self, data: dict, indices: list, insert_mask=None):
    self.data = data
    self.indices = indices
    self.insert_mask = insert_mask

  def __len__(self):
    return len(self.indices)

  def __getitem__(self, idx):
    item = self.indices[idx]

    patches = self.data['insert_patches'][item]

    if self.insert_mask is not None:
      keep = [j for j in range(patches.shape[1]) if j != self.insert_mask]
      patches = patches[:, keep]

    X = (
      patches,
      self.data['iq_level'][item],
      self.data['slice_thickness'][item]
    )

    y = self.data['labels'][item]
    return X, y

In [ ]:
class ResidualBlock(nn.Module):
  def __init__(self, channels: int, dropout: float = 0.0):
    super().__init__()
    self.block = nn.Sequential(
      nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
      nn.BatchNorm2d(channels),
      nn.ReLU(inplace=True),
      nn.Dropout2d(p=dropout),
      nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
      nn.BatchNorm2d(channels),
    )
    self.relu = nn.ReLU(inplace=True)

  def forward(self, x):
    return self.relu(x + self.block(x))


class ResNetIQ(nn.Module):
  """
  Depth 4  -> stem + 1 stage  (1+3 = 4 convs)
  Depth 7  -> stem + 2 stages (1+3+3 = 7 convs)
  Depth 10 -> stem + 3 stages (1+3+3+3 = 10 convs)

  Input: x (B, S, N, 1, 40, 40)
          iq_level (B, 1)
          slice_thick (B, 1)

  Output: (B, 8, 2)
  """

  STAGE_CHANNELS = [32, 64, 128]
  DEPTH_TO_STAGES = {4: 1, 7: 2, 10: 3}

  def __init__(self, depth: int = 10, dropout: float = 0.0):
    super().__init__()

    n_stages = self.DEPTH_TO_STAGES[depth]
    output_channels = self.STAGE_CHANNELS[n_stages - 1]

    self.stem = nn.Sequential(
      nn.Conv2d(1, 16, kernel_size=3, padding=1, bias=False),
      nn.BatchNorm2d(16),
      nn.ReLU(inplace=True),
    )

    stage_definitions = [
      (16, 32),
      (32, 64),
      (64, 128),
    ]
    self.stages = nn.ModuleList([
      nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, stride=2, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
        ResidualBlock(out_ch, dropout),
      )
      for in_ch, out_ch in stage_definitions[:n_stages]
    ])

    self.gap = nn.AdaptiveAvgPool2d((1, 1))

    self.iq_embedding = nn.Sequential(
      nn.Linear(1, 16),
      nn.ReLU(inplace=True)
    )

    self.slice_thick_embedding = nn.Sequential(
      nn.Linear(1, 8),
      nn.ReLU(inplace=True)
    )

    self.head = nn.Linear(output_channels + 16 + 8, 16)

  def forward(self, x: torch.Tensor, iq_level: torch.Tensor, slice_thick: torch.Tensor) -> torch.Tensor:
    B, S, N, C, H, W = x.shape
    x = x.view(B * S * N, C, H, W)

    x = self.stem(x)
    for stage in self.stages:
      x = stage(x)

    x = self.gap(x).flatten(1) # (B*S*N, out_ch)
    x = x.view(B, S, N, -1).mean(dim=2).mean(dim=1) # (B, out_ch)

    iq = self.iq_embedding(iq_level) # (B, 16)
    st = self.slice_thick_embedding(slice_thick) #(B, 8)

    output = self.head(torch.cat([x, iq, st], dim=1))
    output = output.view(B, 8, 2)
    output = self.head(torch.cat([x, iq, st], dim=1))
    return output # (B, 8, 2)

In [32]:
def masked_loss(preds, targets, insert_mask, loss_func):
  """
  For experiment (2) and (3) that an insert is left out optimize only for that insert
  """
  if insert_mask is None:
    return loss_func(preds, targets)

  return loss_func(
    preds[:, insert_mask],
    targets[:, insert_mask]
  )

In [33]:
def train_step(
    model: nn.Module,
    data_loader: DataLoader,
    loss_func: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device = device
  ):

  train_loss = 0
  #Set model in training mode 
  model.train()

  for X, y in data_loader:
    insert_patches, iq_level, slice_thick = X
    insert_patches = insert_patches.to(device)
    iq_level = iq_level.to(device)
    slice_thick = slice_thick.to(device)
    y = y.to(device)

    y_pred = model(insert_patches, iq_level, slice_thick)

    loss = loss_func(y_pred, y)
    train_loss += loss.item()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  train_loss /= len(data_loader)
  print(f"Train loss: {train_loss:.5f}")
  return train_loss

In [34]:
def test_step(
    model: nn.Module,
    data_loader: DataLoader,
    loss_func: nn.Module,
    insert_mask: int = None,
    device: torch.device = device
  ):
  test_loss = 0.0
  #Set model in evaluation mode 
  model.eval()
  predictions = []
  targets = []

  with torch.inference_mode():
    for X, y in data_loader:
      insert_patches, iq_level, slice_thick = X
      insert_patches = insert_patches.to(device)
      iq_level = iq_level.to(device)
      slice_thick = slice_thick.to(device)
      y = y.to(device)

      test_pred = model(insert_patches, iq_level, slice_thick)
      test_loss += masked_loss(test_pred, y, insert_mask, loss_func).item()
      predictions.append(test_pred.cpu())
      targets.append(y.cpu())

    test_loss /= len(data_loader)
    print(f"Test loss: {test_loss:.5f}")
  return test_loss, predictions, targets

In [ ]:
def train_window_split(
    data: dict,
    test_groups: set,
    learning_rate: float,
    epochs: int,
    weight_decay: float,
    optimizer_class,
    depth: int,
    dropout: float,
    window_size: int,
    insert_mask: int | None = None
):
  """
  Train on groups NOT in test_groups, evaluate on test_groups.
  If insert_mask is not None, that insert index is removed from ALL samples.

  Returns: test_loss (float), train_loss (float), history (list),
           predictions (list), targets (list)
  """
  groups = data["acqui_num"].numpy()
  all_indices = np.arange(len(groups))
  train_idx = all_indices[~np.isin(groups, list(test_groups))]
  test_idx = all_indices[ np.isin(groups, list(test_groups))]

  train_dataset = PhantomDataset(data, train_idx, insert_mask=insert_mask)
  test_dataset = PhantomDataset(data, test_idx,  insert_mask=insert_mask)

  train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
  test_loader = DataLoader(test_dataset,  batch_size=1, shuffle=False)

  model = ResNetIQ(depth, dropout).to(device)

  if NUM_GPUS > 1:
    model = nn.DataParallel(model)

  optimizer = optimizer_class(
    model.parameters(), lr=learning_rate, weight_decay=weight_decay
  )
  loss_func = nn.HuberLoss()

  scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.8,
    patience=15,
    min_lr=1e-6,
  )

  history = []
  train_loss = 0.0
  for e in range(epochs):
    train_loss = train_step(model, train_loader, loss_func, optimizer, device)
    scheduler.step(train_loss)
    history.append({
      "window_size": window_size,
      "groups_held_out": [int(g) for g in test_groups],
      "insert_held_out": insert_mask,
      "epoch": int(e),
      "train_loss": float(train_loss),
      "test_loss": None,
    })

  test_loss, predictions, targets = test_step(model, test_loader, loss_func, insert_mask, device)

  for entry in history:
    entry["test_loss"] = float(test_loss)

  del train_loader, test_loader
  del model
  del optimizer
  del scheduler
  torch.cuda.empty_cache()
  gc.collect()

  return history, predictions, targets

In [ ]:
def two_window_held_out(size: int, unique_groups) -> set[int]:
  """
  Groups held out for a given window size in the 2-window variant.
  """
  left = set(unique_groups[1:1 + size])
  right = set(unique_groups[-1 - size:-1])
  return left | right


def three_window_held_out(size: int, unique_groups) -> set[int]:
  """
  Groups held out for a given window size in the 3-window variant.
  """
  core = unique_groups[1:-1]

  left = set(core[:size])

  mid = len(core) // 2
  half = size // 2
  middle = set(
    core[
      max(0, mid - half):
      min(len(core), mid + half + 1)
    ]
  )

  right = set(core[-size:])

  return left | middle | right

In [ ]:
def run_window_variant(
    nd: dict,
    data_name: str,
    exp_tag: str,
    window_fn,
    max_size: int,
    results_dir: str,
    with_insert_mask: bool
) -> None:
  """
  Iterates window sizes 1 -> max_size.
  If with_insert_mask=True, also iterates over every insert index.
  """
  n_inserts   = nd["insert_patches"][0].shape[1]
  groups = nd["acqui_num"].numpy()
  unique_groups = np.unique(groups)
  insert_loop = range(n_inserts) if with_insert_mask else [None]

  for size in range(1, max_size + 1):
    test_groups = window_fn(size, unique_groups)
    group_hash  = hashlib.md5("-".join(map(str, sorted(test_groups))).encode()).hexdigest()[:8]

    for insert_idx in insert_loop:
      insert_str = f"_ins{insert_idx}" if insert_idx is not None else ""
      tag = f"{exp_tag}_{data_name}_size{size:02d}_groups{group_hash}{insert_str}"
      print(
        f"\n[{exp_tag}] size={size}  held-out={sorted(test_groups)}"
        + (f"  masked insert={insert_idx}" if insert_idx is not None else "")
      )

      history, preds, targets = train_window_split(
        nd,
        test_groups = test_groups,
        learning_rate = LEARNING_RATE,
        epochs = EPOCHS,
        weight_decay = WEIGHT_DECAY,
        optimizer_class = OPTIMIZER,
        depth = DEPTH,
        dropout = DROPOUT,
        window_size = size,
        insert_mask = insert_idx,
      )

      save_results( results_dir, tag, history, preds, targets)

      del preds
      del targets
      del history
      gc.collect()

In [38]:
def normalise_data(data: dict, largest_patch_size: int) -> dict:
  new_data = {}
  for key in ["iq_level", "slice_thickness"]:
    mean, std = data[key].mean(), data[key].std()
    new_data[key] = (data[key] - mean) / (std + 1e-8)

  new_data["insert_patches"] = [
    pad_to_size((x - x.mean()) / (x.std() + 1e-8), largest_patch_size)
    for x in data["insert_patches"]
  ]

  label_mean = data["labels"].mean(dim=0)
  label_std = data["labels"].std(dim=0)
  new_data["labels"] = (data["labels"] - label_mean) / (label_std + 1e-8)
  new_data["label_mean"] = label_mean
  new_data["label_std"] = label_std
  new_data["acqui_num"] = data["acqui_num"]
  return new_data

In [ ]:
"""
Experiments:
  (1a) Leave a group out —> 2 windows starting at groups 1 & 23, growing inward
       until only groups 0, 12, 24 remain in training (window sizes 1 -> 11)
  (1b) Leave a group out —> 3 windows starting at groups 1, 12 & 23, growing
       until only groups 0, 8, 16, 24 remain (window sizes 1 -> 7)
  (2a) Same as (1a) but for each window size also leave one insert out at a time
  (2b) Same as (1b) but for each window size also leave one insert out at a time
  (3)  Repeat (2a) and (2b) for each patch-size dataset
"""

def run_experiment_1a(
    data_path: str,
    results_dir: str,
    largest_patch_size: int = LARGEST_PATCH
) -> None:
  """2-window leave-group-out, no insert masking. Window sizes 1–11."""
  nd = normalise_data(torch.load(data_path), largest_patch_size)
  run_window_variant(
    nd, Path(data_path).stem, "Exp1a",
    two_window_held_out, TWO_WINDOW_MAX_SIZE,
    results_dir, with_insert_mask=False,
  )

def run_experiment_1b(
    data_path: str,
    results_dir: str,
    largest_patch_size: int = LARGEST_PATCH
) -> None:
  """3-window leave-group-out, no insert masking. Window sizes 1–7."""
  nd = normalise_data(torch.load(data_path), largest_patch_size)
  run_window_variant(
    nd, Path(data_path).stem, "Exp1b",
    three_window_held_out, THREE_WINDOW_MAX_SIZE,
    results_dir, with_insert_mask=False,
  )

def run_experiment_2a(
    data_path: str,
    results_dir: str,
    largest_patch_size: int = LARGEST_PATCH
) -> None:
  """2-window leave-group-out + leave-one-insert-out. Window sizes 1–11."""
  nd = normalise_data(torch.load(data_path), largest_patch_size)
  run_window_variant(
    nd, Path(data_path).stem, "Exp2a",
    two_window_held_out, TWO_WINDOW_MAX_SIZE,
    results_dir, with_insert_mask=True,
  )

def run_experiment_2b(
    data_path: str,
    results_dir: str,
    largest_patch_size: int = LARGEST_PATCH
) -> None:
  """3-window leave-group-out + leave-one-insert-out. Window sizes 1–7."""
  nd = normalise_data(torch.load(data_path), largest_patch_size)
  run_window_variant(
    nd, Path(data_path).stem, "Exp2b",
    three_window_held_out, THREE_WINDOW_MAX_SIZE,
    results_dir, with_insert_mask=True,
  )

def run_experiment_3(
    patch_data_paths: list[str],
    patch_sizes: list[int],
    results_dir: str
) -> None:
  """
  Re-runs Exp 2a and 2b for each (data_path, patch_size) pair.
  """
  for data_path, patch_size in zip(patch_data_paths, patch_sizes):
    sub_dir = os.path.join(results_dir, f"patch{patch_size}")
    run_experiment_2a(data_path, os.path.join(sub_dir, "2a"), patch_size)
    run_experiment_2b(data_path, os.path.join(sub_dir, "2b"), patch_size)

In [ ]:
DATA = "./catphan_rois_exp1_2_40.pt"

run_experiment_1a(DATA, "./results_exp1a")
run_experiment_1b(DATA, "./results_exp1b")
run_experiment_2a(DATA, "./results_exp2a")
run_experiment_2b(DATA, "./results_exp2b")

run_experiment_3(
  patch_data_paths=[
    "./cataphan_rois_exp3_60.pt",
    "./cataphan_rois_exp3_80.pt",
    "./cataphan_rois_exp3_100.pt"
  ],
  patch_sizes=[60, 80, 100],
  results_dir="./results_exp3"
)